# Security posture

**How fast are we closing risk, how much is open right now, and is it getting worse?**

The leadership view. One number, what it is measured over, and where the open work sits.
Nothing here is a chart: if a figure needs a chart to be legible it belongs on one of the
other notebooks, and this page is the one someone reads in a meeting.

## How to read this notebook

Every cell answers one question and shows one thing. Run them in order the first time; after
that any cell can be re-run on its own.

**Set the widgets at the top before you run anything.** `catalog` has no default on purpose.
Set the notebook to **Run accessed commands** (the dropdown beside *Run all*) if you want a
widget change to re-run the cells that depend on it — otherwise you will change the filter and
read a chart drawn under the old one.

Everything here reads one scan, pinned in cell 1. Charts that span scans say so in their title.

In [ ]:
PAGE = {"group_by": ("subscription_name", panels.GROUP_DIMENSIONS)}

import os, sys

_paths = []
try:
    _paths.append(dbutils.widgets.get("module_path"))
except Exception:  # noqa: BLE001 -- the widget does not exist yet on a first run
    pass
_here = os.getcwd()
_paths += [_here, os.path.dirname(_here)]
for _p in _paths:
    if _p and os.path.exists(os.path.join(_p, "panels.py")):
        sys.path.insert(0, _p)
        break
else:
    raise RuntimeError("brick modules are not on sys.path -- see brick/README.md, step 2")

import panels, figures, tiles

panels.declare_widgets(**PAGE)
ctx = panels.context(spark, **{name: str(spec[0]) for name, spec in PAGE.items()})
displayHTML(tiles.scan_zone_from(panels.last_scan(spark, ctx).first()))

## Median time to remediate

Kaplan–Meier, not a median of what closed. Averaging over resolved findings only is
survivorship bias with a respectable name — the slowest findings are disproportionately
the ones *still open*, so excluding them flatters the number exactly when a programme is
falling behind. Still-open findings stay in as right-censored observations.

A `> 90d` reading is not a missing number: it means survival never fell to 50%, so more
than half the register is still open and the median is a floor.

In [ ]:
displayHTML(
    tiles.posture_hero(
        panels.posture(spark, ctx).first(),
        panels.week_delta(spark, ctx).first(),
    )
)

## Open vulnerabilities

Counts as of the pinned scan. A zero is shown rather than hidden — *no criticals* is the
most reassuring thing this page can say, and it can only say it if the tile is there.

In [ ]:
displayHTML(
    tiles.severity_tiles(
        [r.asDict() for r in panels.severity_open(spark, ctx).collect()],
        sub="Open findings per severity, as of the scan above.",
    )
)

## Where the slow work is

GAS breaks this down by *value chain domain*. brick has no domain rules and does not
ingest asset tags, so the nearest real dimension is the one below — a **billing**
boundary that often, but not always, maps to an owning team. Read it as *which account
drags the headline up*, not as *which team is slow*.

Change the `group_by` widget to slice it another way. `04_scan_history` has the trend and
`01_mttr_sla` has the same table with SLA columns and a contribution chart.

In [ ]:
display(panels.km_by(spark, ctx, ctx.param('group_by'), top_n=5))

---

**Next:** `01_mttr_sla` for where remediation is slow · `02_program_performance` for
whether the effort landed on the right findings · `06_run_and_verify` to run a scan.